# One-Class SVM for Anomaly Detection

## What This Notebook Covers
This notebook implements **One-Class SVM** for anomaly detection — both a simplified from-scratch version using NumPy (with scipy for optimisation) and using scikit-learn's `OneClassSVM` class. One-Class SVM learns a decision boundary that encloses normal data; points outside are anomalies.

## Prerequisites
- Support Vector Machines (margin, kernel trick)
- Quadratic programming / optimisation basics
- RBF (Radial Basis Function) kernel
- Python, NumPy, pandas, matplotlib

## Dataset
**Credit Card Fraud Detection** (Kaggle)  
Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
284,807 transactions with 30 features (V1-V28 from PCA + Time + Amount). Target: `Class` (0=normal, 1=fraud, ~0.17% fraud rate).

## Credits
- Dataset by ULB Machine Learning Group
- Schölkopf, Platt, Shawe-Taylor, Smola, Williamson (2001) — Original One-Class SVM paper
- Gradientts ML Intern Program — 2026

In [ ]:
# ============================================================
# Cell 2: All Imports
# ============================================================

import numpy as np                   # Core numerical computing
import pandas as pd                  # Data manipulation and loading
import matplotlib.pyplot as plt      # Plotting and visualisation
import seaborn as sns                # Statistical visualisation
from sklearn.preprocessing import StandardScaler  # Feature scaling (critical for SVM)
from sklearn.model_selection import train_test_split  # Splitting
from sklearn.svm import OneClassSVM   # Sklearn One-Class SVM
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve
)
from scipy.optimize import minimize   # For the from-scratch QP optimisation

# Reproducibility
np.random.seed(42)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports successful.')

## Part 1: Theory Recap

**One-Class SVM** learns a decision boundary around normal data by solving an optimisation problem in kernel space:

1. **Core idea:** Map data to a high-dimensional kernel space (via RBF kernel), find a hyperplane that separates data from the origin with maximum margin.
2. **Decision function:** $f(x) = \text{sign}(\sum_i \alpha_i K(x_i, x) - \rho)$. Points inside boundary get +1 (normal), outside get -1 (anomaly).
3. **ν (nu) parameter:** Upper bound on outlier fraction AND lower bound on support vector fraction. ν=0.05 means ≤5% of training data can be outliers.
4. **RBF kernel:** $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$. γ controls boundary tightness (small γ = smooth, large γ = tight).
5. **Key limitation:** Training is O(n²)–O(n³) due to the kernel matrix, making it unsuitable for large datasets.

## Loading the Dataset

We load the Credit Card Fraud dataset and create a small subsample (One-Class SVM is O(n²)–O(n³) and cannot handle large datasets).

In [ ]:
# ============================================================
# Cell 4: Load the real-world dataset
# ============================================================

try:
    df = pd.read_csv('creditcard.csv')
    print('Loaded from local file.')
except FileNotFoundError:
    from sklearn.datasets import fetch_openml
    print('Local file not found. Downloading from OpenML...')
    credit = fetch_openml(data_id=1597, as_frame=True, parser='auto')
    df = credit.frame
    if 'Class' not in df.columns:
        df = df.rename(columns={df.columns[-1]: 'Class'})
    df['Class'] = df['Class'].astype(int)
    print('Downloaded successfully.')

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:\n{df["Class"].value_counts()}')
print(f'\nFraud rate: {df["Class"].mean()*100:.3f}%')

print('\n--- First 5 rows ---')
df.head()

In [ ]:
print('--- Dataset Info ---')
print(df.info())
print('\n--- Statistical Summary ---')
df.describe()

## Preprocessing

Feature scaling is **critical** for SVM (RBF kernel uses Euclidean distances — unscaled features dominate). We use a small subsample due to SVM's O(n²) training complexity.

In [ ]:
# ============================================================
# Cell 5: Preprocessing
# ============================================================

# Step 1: Scale Time and Amount
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = scaler.fit_transform(df[['Time']])
df_clean = df.drop(['Time', 'Amount'], axis=1)

# Step 2: Check for nulls
print(f'Null values: {df_clean.isnull().sum().sum()}')

# Step 3: Small subsample for SVM (O(n^2) to O(n^3) training)
fraud = df_clean[df_clean['Class'] == 1].sample(n=200, random_state=42)  # Sample of fraud
normal = df_clean[df_clean['Class'] == 0].sample(n=2000, random_state=42)
df_sub = pd.concat([normal, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(f'Class distribution:\n{df_sub["Class"].value_counts()}')

# Step 4: Select features and scale
feature_cols = ['V1', 'V2', 'V3', 'V4', 'V10', 'V11', 'V14', 'V17', 'scaled_Amount']
X_sub = df_sub[feature_cols].values
y_sub = df_sub['Class'].values

# INTERVIEW NOTE: Feature scaling is CRITICAL for SVM.
# RBF kernel: K(x,y) = exp(-gamma * ||x-y||^2)
# Without scaling, features with large range dominate the kernel.
scaler_sub = StandardScaler()
X_sub = scaler_sub.fit_transform(X_sub)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)

# For One-Class SVM: train on NORMAL data only
X_train_normal = X_train[y_train == 0]

print(f'\nTraining set (normal only): {X_train_normal.shape}')
print(f'Test set: {X_test.shape} (fraud: {y_test.sum()}, normal: {(y_test==0).sum()})')

## Part 2: From Scratch Implementation

We implement a simplified One-Class SVM from scratch.

**What we are building:** A `OneClassSVMScratch` class that:
1. Computes the RBF kernel matrix between training points
2. Solves a simplified optimisation to find dual coefficients (α)
3. Uses support vectors to compute the decision function for new points

**Why from scratch?** Understanding the kernel matrix computation, the role of ν in bounding outlier fraction, and how support vectors define the decision boundary is essential for SVM interviews.

**Note:** A full SMO solver is very complex. We use `scipy.optimize.minimize` for the quadratic programming step, keeping the focus on the SVM formulation rather than solver internals.

In [ ]:
# ============================================================
# Cell 7: From-Scratch One-Class SVM (NumPy + scipy.optimize)
# ============================================================

class OneClassSVMScratch:
    """
    Simplified One-Class SVM using RBF kernel.
    Solves the dual formulation using scipy.optimize.
    
    The dual problem:
        min  0.5 * alpha^T * K * alpha
        s.t. 0 <= alpha_i <= 1/(nu*n)
             sum(alpha) = 1
    """
    
    def __init__(self, nu=0.05, gamma='scale'):
        # INTERVIEW NOTE: nu has a dual interpretation:
        # 1. Upper bound on the fraction of outliers
        # 2. Lower bound on the fraction of support vectors
        # Setting nu=0.05 means at most 5% outliers in training data.
        self.nu = nu
        self.gamma_param = gamma
        self.alpha = None
        self.rho = None
        self.support_vectors = None
        self.support_alpha = None
    
    def _compute_gamma(self, X):
        """Compute gamma for the RBF kernel."""
        if self.gamma_param == 'scale':
            # INTERVIEW NOTE: 'scale' = 1 / (n_features * X.var())
            # This is sklearn's default and normalises by data spread.
            return 1.0 / (X.shape[1] * X.var())
        elif self.gamma_param == 'auto':
            return 1.0 / X.shape[1]
        else:
            return self.gamma_param
    
    def _rbf_kernel(self, X1, X2):
        """
        Compute RBF kernel matrix between X1 and X2.
        K(x, y) = exp(-gamma * ||x - y||^2)
        """
        # INTERVIEW NOTE: The RBF kernel maps data to infinite-dimensional space.
        # Small gamma = large radius = smooth boundary (underfitting)
        # Large gamma = small radius = tight boundary (overfitting)
        sq1 = np.sum(X1 ** 2, axis=1)
        sq2 = np.sum(X2 ** 2, axis=1)
        dist_sq = sq1[:, None] + sq2[None, :] - 2 * X1 @ X2.T
        dist_sq = np.maximum(dist_sq, 0)
        return np.exp(-self.gamma * dist_sq)
    
    def fit(self, X):
        """
        Fit the One-Class SVM on normal training data.
        Solves the dual optimisation problem.
        """
        self.X_train = np.array(X, dtype=np.float64)
        n = len(X)
        self.gamma = self._compute_gamma(X)
        
        # Step 1: Compute kernel matrix
        # INTERVIEW NOTE: Kernel matrix is O(n^2) in memory and computation.
        # This is the main scalability bottleneck of kernel SVMs.
        K = self._rbf_kernel(X, X)
        
        # Step 2: Solve dual problem using scipy
        # Objective: min 0.5 * alpha^T * K * alpha
        upper_bound = 1.0 / (self.nu * n)
        
        def objective(alpha):
            return 0.5 * alpha @ K @ alpha
        
        def gradient(alpha):
            return K @ alpha
        
        # Constraints: sum(alpha) = 1
        constraints = {'type': 'eq', 'fun': lambda a: np.sum(a) - 1.0}
        
        # Bounds: 0 <= alpha_i <= 1/(nu*n)
        bounds = [(0, upper_bound) for _ in range(n)]
        
        # Initial guess: uniform
        alpha0 = np.ones(n) / n
        
        # Solve
        result = minimize(objective, alpha0, jac=gradient, bounds=bounds,
                         constraints=constraints, method='SLSQP',
                         options={'maxiter': 500, 'ftol': 1e-8})
        
        self.alpha = result.x
        
        # Step 3: Find support vectors (alpha > small threshold)
        # INTERVIEW NOTE: Support vectors are points with alpha > 0.
        # They lie ON the decision boundary.
        sv_threshold = 1e-6
        sv_mask = self.alpha > sv_threshold
        self.support_vectors = self.X_train[sv_mask]
        self.support_alpha = self.alpha[sv_mask]
        
        # Step 4: Compute rho (offset) using support vectors
        # For support vectors with 0 < alpha < upper_bound:
        # rho = w · phi(x_sv) = sum_i alpha_i * K(x_i, x_sv)
        free_sv_mask = sv_mask & (self.alpha < upper_bound - sv_threshold)
        if np.any(free_sv_mask):
            free_sv_idx = np.where(free_sv_mask)[0]
            rho_values = []
            for idx in free_sv_idx:
                rho_val = np.sum(self.alpha * K[:, idx])
                rho_values.append(rho_val)
            self.rho = np.mean(rho_values)
        else:
            # Fallback: use all support vectors
            K_sv = self._rbf_kernel(self.X_train, self.support_vectors)
            decision_vals = self.alpha @ K_sv
            self.rho = np.mean(decision_vals)
        
        print(f'  Optimisation converged: {result.success}')
        print(f'  Number of support vectors: {len(self.support_vectors)}')
        print(f'  Support vector fraction: {len(self.support_vectors)/n:.3f}')
        print(f'  Rho (offset): {self.rho:.4f}')
        
        return self
    
    def decision_function(self, X):
        """
        Compute the decision function for new points.
        f(x) = sum_i alpha_i * K(x_i, x) - rho
        Positive = normal (inside boundary), Negative = anomaly (outside).
        """
        # INTERVIEW NOTE: Only support vectors contribute to the decision function.
        # This is the "sparsity" advantage of SVMs.
        K_new = self._rbf_kernel(self.X_train, np.array(X))
        decision = self.alpha @ K_new - self.rho
        return decision
    
    def predict(self, X):
        """Predict labels: +1 = normal, -1 = anomaly."""
        decision = self.decision_function(X)
        return np.where(decision >= 0, 1, -1)


print('OneClassSVMScratch class defined successfully.')

## Fitting and Evaluating the From-Scratch One-Class SVM

We fit on **normal training data only** (this is the "one class" — we only model normal behaviour). Anomalies are points outside the learned boundary.

In [ ]:
# ============================================================
# Cell 8: Fit scratch model and evaluate
# ============================================================

# Fit the scratch One-Class SVM on normal data
print('Fitting One-Class SVM from scratch (this may take a moment)...')
oc_svm_scratch = OneClassSVMScratch(nu=0.05, gamma='scale')
oc_svm_scratch.fit(X_train_normal)

# Score and predict on test set
scratch_decision = oc_svm_scratch.decision_function(X_test)
scratch_preds = oc_svm_scratch.predict(X_test)
scratch_labels = np.where(scratch_preds == -1, 1, 0)

print('\n--- From-Scratch One-Class SVM Results ---')
print(classification_report(y_test, scratch_labels, target_names=['Normal', 'Fraud']))

# AUC (more negative decision = more anomalous)
scratch_auc = roc_auc_score(y_test, -scratch_decision)
print(f'AUC-ROC: {scratch_auc:.4f}')

scratch_ap = average_precision_score(y_test, -scratch_decision)
print(f'Average Precision (PR-AUC): {scratch_ap:.4f}')

## Part 3: Sklearn Implementation

Scikit-learn's `OneClassSVM` uses the highly optimised **libsvm** library under the hood, providing:
- Efficient SMO (Sequential Minimal Optimization) solver
- Multiple kernel options (rbf, linear, poly, sigmoid)
- `decision_function()` for continuous anomaly scoring

Key difference from scratch: sklearn's libsvm is orders of magnitude faster and more numerically stable.

In [ ]:
# ============================================================
# Cell 10: Sklearn One-Class SVM
# ============================================================

# Fit sklearn One-Class SVM on normal training data
sklearn_ocsvm = OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=0.05
)
sklearn_ocsvm.fit(X_train_normal)

# Predict and score
sklearn_preds = sklearn_ocsvm.predict(X_test)
sklearn_decision = sklearn_ocsvm.decision_function(X_test)
sklearn_labels = np.where(sklearn_preds == -1, 1, 0)

print('--- Sklearn One-Class SVM Results ---')
print(classification_report(y_test, sklearn_labels, target_names=['Normal', 'Fraud']))

sklearn_auc = roc_auc_score(y_test, -sklearn_decision)
print(f'AUC-ROC: {sklearn_auc:.4f}')

sklearn_ap = average_precision_score(y_test, -sklearn_decision)
print(f'Average Precision (PR-AUC): {sklearn_ap:.4f}')

# --- Direct Comparison ---
print('\n=== Scratch vs Sklearn Comparison ===')
print(f'{"Metric":<25} {"Scratch":<15} {"Sklearn":<15}')
print(f'{"AUC-ROC":<25} {scratch_auc:<15.4f} {sklearn_auc:<15.4f}')
print(f'{"Avg Precision (PR-AUC)":<25} {scratch_ap:<15.4f} {sklearn_ap:<15.4f}')
print(f'{"Support Vectors":<25} {len(oc_svm_scratch.support_vectors):<15d} {len(sklearn_ocsvm.support_vectors_):<15d}')

## Visualisations

1. **Decision Function Score Distribution** — normal vs fraud
2. **ROC Curve** — scratch vs sklearn comparison

In [ ]:
# ============================================================
# Cell 11: Visualisations
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Decision Function Distribution ---
ax1 = axes[0]
ax1.hist(sklearn_decision[y_test == 0], bins=50, alpha=0.7, label='Normal', color='steelblue', density=True)
ax1.hist(sklearn_decision[y_test == 1], bins=50, alpha=0.7, label='Fraud', color='crimson', density=True)
ax1.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Decision Boundary (f=0)')
ax1.set_xlabel('Decision Function Value', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('One-Class SVM: Decision Function Distribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# --- Plot 2: ROC Curve ---
ax2 = axes[1]
fpr_s, tpr_s, _ = roc_curve(y_test, -scratch_decision)
fpr_sk, tpr_sk, _ = roc_curve(y_test, -sklearn_decision)
ax2.plot(fpr_s, tpr_s, label=f'Scratch (AUC={scratch_auc:.3f})', color='darkorange', linewidth=2)
ax2.plot(fpr_sk, tpr_sk, label=f'Sklearn (AUC={sklearn_auc:.3f})', color='steelblue', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve: Scratch vs Sklearn OC-SVM', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

# --- Plot 3: Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))
prec, rec, _ = precision_recall_curve(y_test, -sklearn_decision)
ax.plot(rec, prec, color='steelblue', linewidth=2, label=f'OC-SVM (AP={sklearn_ap:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve (One-Class SVM)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

Key hyperparameters for One-Class SVM:

1. **`nu`** — controls the trade-off between outlier fraction and support vector count. Higher ν = looser boundary, more outliers allowed.
2. **`gamma`** — controls RBF kernel width. Smaller γ = smoother boundary; larger γ = tighter boundary.

We vary both and measure AUC-ROC.

In [ ]:
# ============================================================
# Cell 13: Hyperparameter Experiments
# ============================================================

# --- Experiment 1: nu effect ---
nu_values = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
nu_aucs = []

for nu_val in nu_values:
    model = OneClassSVM(kernel='rbf', gamma='scale', nu=nu_val)
    model.fit(X_train_normal)
    scores = model.decision_function(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    n_sv = len(model.support_vectors_)
    nu_aucs.append(auc_val)
    print(f'nu={nu_val:<5.2f}  AUC-ROC={auc_val:.4f}  SVs={n_sv}')

# --- Experiment 2: gamma effect ---
gamma_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
gamma_aucs = []
best_nu = nu_values[np.argmax(nu_aucs)]

for g in gamma_values:
    model = OneClassSVM(kernel='rbf', gamma=g, nu=best_nu)
    model.fit(X_train_normal)
    scores = model.decision_function(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    gamma_aucs.append(auc_val)
    print(f'gamma={g:<6.3f}  AUC-ROC={auc_val:.4f}')

# --- Plot results ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(nu_values, nu_aucs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(x=best_nu, color='crimson', linestyle='--', label=f'Best ν={best_nu}')
axes[0].set_xlabel('ν (nu)', fontsize=12)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('Effect of ν on OC-SVM Performance', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

axes[1].plot(gamma_values, gamma_aucs, 's-', color='coral', linewidth=2, markersize=8)
axes[1].set_xlabel('γ (gamma)', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title(f'Effect of γ on OC-SVM (ν={best_nu})', fontsize=13, fontweight='bold')
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Key Conceptual Question: *"Explain the relationship between the ν parameter and the number of support vectors in One-Class SVM."*

**Answer Structure:**

1. **ν as a dual controller:** The ν parameter simultaneously controls two things:
   - **Upper bound on outlier fraction** — at most ν × n training points can lie outside (or on) the boundary.
   - **Lower bound on support vector fraction** — at least ν × n training points will be support vectors.

2. **Why this duality?** In the optimisation, ν appears in the upper bound of dual variables: $0 \leq \alpha_i \leq \frac{1}{\nu n}$. When ν is small, each α can be large → fewer points needed to reach sum(α)=1 → fewer support vectors but tighter boundary. When ν is large, each α is small → more points needed → more support vectors but looser boundary.

3. **Practical implication:** Setting ν = 0.05 is saying: "I expect at most 5% of my training data to be outliers, and I'm willing to use at least 5% of my data as support vectors." If your data is truly clean (no outliers in training), a small ν gives a tight, precise boundary.

4. **Connection to standard SVM:** In binary SVM, C controls the trade-off between margin width and misclassification. In One-Class SVM, ν replaces C with a more interpretable control that directly bounds the outlier fraction.

5. **Choosing ν in practice:** Start with ν equal to your estimated contamination rate. If unknown, try ν ∈ [0.01, 0.2] and select based on cross-validated performance metrics.

## Key Takeaways

1. **One-Class SVM learns a decision *boundary*, not a density** — it finds the tightest boundary in kernel space that encloses normal data. Points outside are anomalies. This makes it complementary to density-based methods.

2. **The ν parameter is uniquely interpretable** — it directly bounds both the outlier fraction and support vector fraction, making it easier to set than traditional SVM's C parameter.

3. **Feature scaling is non-negotiable** — the RBF kernel uses Euclidean distances. Without StandardScaler, features with large ranges completely dominate the kernel matrix. Always scale before fitting.

4. **Training is O(n²) to O(n³)** — the kernel matrix computation and QP solver make OC-SVM impractical for datasets larger than ~10K samples. For large data, use Isolation Forest or SGD-based approaches.

5. **RBF kernel's γ controls boundary tightness** — small γ creates a smooth, general boundary (may miss subtle anomalies); large γ creates a tight, complex boundary (may overfit to noise). Use `gamma='scale'` as a sensible default.